In [1]:
# ============================================================
# CELL 1: Imports, Config & Manual .env Loader
# ============================================================

import os
import sys
import json
import re
import time
import logging
from pathlib import Path
from typing import Dict, List, Any, Optional

# ── Third-party imports ─────────────────────────────────────
try:
    import chromadb
    print("✅ chromadb imported")
except ImportError:
    print("❌ chromadb not found — pip install chromadb"); sys.exit(1)

try:
    from sentence_transformers import SentenceTransformer
    print("✅ sentence_transformers imported")
except ImportError:
    print("❌ sentence_transformers not found — pip install sentence-transformers"); sys.exit(1)

try:
    import google.generativeai as genai
    print("✅ google-generativeai imported")
except ImportError:
    print("❌ google-generativeai not found — pip install google-generativeai"); sys.exit(1)

try:
    import torch
    print(f"✅ torch imported  (CUDA: {torch.cuda.is_available()})")
except ImportError:
    print("⚠️  torch not found — sentence-transformers will use CPU")
    torch = None

# ── Manual .env.query parser (no python-dotenv dependency) ──
def _load_env_file(filepath: str) -> Dict[str, str]:
    """Parse a .env style file manually."""
    env_vars: Dict[str, str] = {}
    try:
        with open(filepath, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line or line.startswith("#"):
                    continue
                if "=" in line:
                    key, _, value = line.partition("=")
                    key   = key.strip()
                    value = value.strip().strip('"').strip("'")
                    env_vars[key] = value
    except FileNotFoundError:
        print(f"⚠️  .env file not found at: {filepath}")
    except Exception as exc:
        print(f"⚠️  Could not read .env file: {exc}")
    return env_vars

# Change this line in your Cell 1:
_env_path = Path(".env.query")
_env_vars  = _load_env_file(str(_env_path))

# ── Constants ────────────────────────────────────────────────
API_KEY    = _env_vars.get("QUERY_KEY", os.environ.get("QUERY_KEY", ""))
DB_PATH    = "database/chroma_db"
MODEL_NAME = "gemini-2.5-flash"
EMBED_MODEL= "all-MiniLM-L6-v2"
MAX_RETRIES= 2

if API_KEY:
    _masked = API_KEY[:6] + "..." + API_KEY[-4:] if len(API_KEY) > 10 else "****"
    print(f"✅ API key loaded: {_masked}")
else:
    print("⚠️  QUERY_KEY not found in .env.query — LLM disabled, regex fallback active")

# ── Logging ──────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)],
)
logger = logging.getLogger("MultiProductAgent")
logging.getLogger("sentence_transformers").setLevel(logging.WARNING)
logging.getLogger("chromadb").setLevel(logging.WARNING)

# ── GeminiModelManager singleton ─────────────────────────────
class GeminiModelManager:
    """Lazy singleton wrapper around google-generativeai."""
    _instance = None
    _model    = None

    def __new__(cls):
        if cls._instance is None:
            cls._instance = super().__new__(cls)
        return cls._instance

    def get_model(self, api_key: str = "", model_name: str = MODEL_NAME):
        if self._model is None and api_key:
            try:
                genai.configure(api_key=api_key)
                self._model = genai.GenerativeModel(
                    model_name,
                    system_instruction=(
                        "You are a helpful product recommendation assistant. "
                        "Always respond with valid JSON when asked to classify queries "
                        "or format recommendations."
                    ),
                )
                logger.info(f"GeminiModelManager: model '{model_name}' ready")
            except Exception as exc:
                logger.warning(f"GeminiModelManager: could not create model — {exc}")
                self._model = None
        return self._model

_gemini_manager = GeminiModelManager()

print("\n✅ Cell 1 complete")
print(f"   DB_PATH     : {DB_PATH}")
print(f"   MODEL_NAME  : {MODEL_NAME}")
print(f"   EMBED_MODEL : {EMBED_MODEL}")
print(f"   MAX_RETRIES : {MAX_RETRIES}")


✅ chromadb imported


C:\Users\pmthe\product-gpt\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ sentence_transformers imported
✅ google-generativeai imported
✅ torch imported  (CUDA: False)
✅ API key loaded: AQ.Ab8...PkHw

✅ Cell 1 complete
   DB_PATH     : database/chroma_db
   MODEL_NAME  : gemini-2.5-flash
   EMBED_MODEL : all-MiniLM-L6-v2
   MAX_RETRIES : 2


C:\Users\pmthe\AppData\Local\Temp\ipykernel_8020\1857446424.py:28: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [2]:
import os
from pathlib import Path

# Print where Python thinks it is
print(f"Current working directory: {os.getcwd()}")

# Check if the file exists at the expected path
env_path = Path(".env.query")
print(f"Checking for .env.query at: {env_path.absolute()}")
print(f"Does it exist? {env_path.exists()}")

Current working directory: C:\Users\pmthe\product-gpt
Checking for .env.query at: C:\Users\pmthe\product-gpt\.env.query
Does it exist? True


In [3]:
# ============================================================
# CELL 2: Self-Contained ProductRetriever
# ============================================================

class ProductRetriever:
    """
    Retrieves products from ChromaDB using semantic search.
    ONE collection named 'products' with product_type in metadata.
    """

    SUPPORTED_TYPES = ["laptop", "phone", "tablet", "headphone", "smartwatch"]

    # Ordered keyword map for type detection (specific first)
    _TYPE_KEYWORDS = [
        ("headphone", [r"headphone", r"earphone", r"earbuds", r"tws\b",
                       r"headset", r"airpods", r"wf-", r"wh-", r"\bbuds\b"]),
        ("tablet",    [r"tablet", r"ipad", r"galaxy tab"]),
        ("smartwatch",[r"smartwatch", r"smart watch", r"fitbit",
                       r"galaxy watch", r"apple watch", r"\bwatch\b"]),
        ("laptop",    [r"laptop", r"notebook", r"chromebook", r"macbook",
                       r"ultrabook", r"thinkpad", r"ideapad"]),
        ("phone",     [r"phone", r"smartphone", r"mobile", r"iphone",
                       r"android", r"oneplus", r"redmi", r"poco",
                       r"realme", r"pixel", r"\bgalaxy\b"]),
    ]

    def __init__(self, db_path: str = DB_PATH):
        self.db_path = db_path
        logger.info(f"ProductRetriever: connecting to ChromaDB at '{db_path}'")

        # ChromaDB client
        try:
            self._client = chromadb.PersistentClient(path=db_path)
            self._collection = self._client.get_collection("products")
            logger.info(f"ProductRetriever: collection 'products' loaded "
                        f"({self._collection.count()} items)")
        except Exception as exc:
            logger.error(f"ProductRetriever: ChromaDB error — {exc}")
            raise

        # Embedding model
        device = "cuda" if (torch is not None and torch.cuda.is_available()) else "cpu"
        logger.info(f"ProductRetriever: loading '{EMBED_MODEL}' on {device}")
        self._embed_model = SentenceTransformer(EMBED_MODEL, device=device)
        logger.info("ProductRetriever: embedding model ready")

    # ── helpers ──────────────────────────────────────────────

    @staticmethod
    def _fix_price(meta: Dict[str, Any]) -> Dict[str, Any]:
        """Convert string/float price field to float in-place."""
        if "price" in meta:
            try:
                raw = str(meta["price"]).replace(",", "").replace("₹", "").strip()
                meta["price"] = float(raw)
            except (ValueError, TypeError):
                meta["price"] = 0.0
        return meta

    def _embed(self, text: str):
        return self._embed_model.encode(
            text, normalize_embeddings=True, convert_to_numpy=True
        ).tolist()

    def _build_results(self, qr) -> List[Dict[str, Any]]:
        """Convert raw ChromaDB query result to list of dicts."""
        results = []
        ids       = qr.get("ids",       [[]])[0]
        metas     = qr.get("metadatas", [[]])[0]
        docs      = qr.get("documents", [[]])[0]
        distances = qr.get("distances", [[]])[0]

        for i, rid in enumerate(ids):
            meta  = self._fix_price(dict(metas[i] or {}))
            dist  = distances[i] if i < len(distances) else 1.0
            score = max(0.0, min(1.0, 1.0 - dist))
            results.append({
                "id":       rid,
                "metadata": meta,
                "document": docs[i] if i < len(docs) else "",
                "score":    round(score, 4),
            })
        return results

    # ── public API ───────────────────────────────────────────

    def search(
        self,
        query:        str,
        n_results:    int  = 5,
        product_type: Optional[str] = None,
        min_score:    float = 0.0,
    ) -> List[Dict[str, Any]]:
        """Semantic search against the 'products' collection."""
        embedding = self._embed(query)
        kwargs: Dict[str, Any] = {
            "query_embeddings": [embedding],
            "n_results":        n_results,
            "include":          ["metadatas", "documents", "distances"],
        }
        if product_type and product_type in self.SUPPORTED_TYPES:
            kwargs["where"] = {"product_type": {"$eq": product_type}}

        try:
            raw = self._collection.query(**kwargs)
        except Exception as exc:
            logger.error(f"ProductRetriever.search error: {exc}")
            return []

        results = self._build_results(raw)
        if min_score > 0:
            results = [r for r in results if r["score"] >= min_score]
        return results

    def filter_by_price(
        self,
        query:        str,
        max_price:    float,
        n_results:    int  = 10,
        product_type: Optional[str] = None,
    ) -> List[Dict[str, Any]]:
        """Search then post-filter by max price (ChromaDB float filter unreliable)."""
        # Fetch more candidates, then filter
        candidates = self.search(query, n_results=max(n_results * 3, 20),
                                 product_type=product_type)
        filtered = [r for r in candidates
                    if r["metadata"].get("price", float("inf")) <= max_price]
        return filtered[:n_results]

    def filter_by_brand(
        self,
        query:        str,
        brand:        str,
        n_results:    int  = 5,
        product_type: Optional[str] = None,
    ) -> List[Dict[str, Any]]:
        """Search then post-filter by brand name (case-insensitive)."""
        candidates = self.search(query, n_results=max(n_results * 3, 20),
                                 product_type=product_type)
        brand_lower = brand.lower()
        filtered = [r for r in candidates
                    if r["metadata"].get("brand", "").lower() == brand_lower]
        return filtered[:n_results]

    def detect_product_type(self, query: str) -> Optional[str]:
        """Return detected product type keyword or None."""
        q = query.lower()
        for ptype, patterns in self._TYPE_KEYWORDS:
            for pat in patterns:
                if re.search(pat, q):
                    return ptype
        return None

    def get_available_types(self) -> List[str]:
        return list(self.SUPPORTED_TYPES)

    def get_product_count(self) -> int:
        return self._collection.count()


print("✅ Cell 2 complete  —  ProductRetriever defined")


✅ Cell 2 complete  —  ProductRetriever defined


In [4]:
# ============================================================
# CELL 3: Self-Contained QueryClassifier
# ============================================================

class QueryClassifier:
    """
    Classify user queries with LLM (Gemini) + regex fallback.
    Returns product_type, max_price, features, confidence, method.
    """

    SUPPORTED_TYPES = ["laptop", "phone", "tablet", "headphone", "smartwatch"]

    _TYPE_KEYWORDS = [
        ("headphone", [r"headphone", r"earphone", r"earbuds", r"tws\b",
                       r"headset", r"airpods", r"wf-", r"wh-", r"\bbuds\b"]),
        ("tablet",    [r"tablet", r"ipad", r"galaxy tab"]),
        ("smartwatch",[r"smartwatch", r"smart watch", r"fitbit",
                       r"galaxy watch", r"apple watch", r"\bwatch\b"]),
        ("laptop",    [r"laptop", r"notebook", r"chromebook", r"macbook",
                       r"ultrabook", r"thinkpad", r"ideapad"]),
        ("phone",     [r"phone", r"smartphone", r"mobile", r"iphone",
                       r"android", r"oneplus", r"redmi", r"poco",
                       r"realme", r"pixel", r"\bgalaxy\b"]),
    ]

    _FEATURE_KEYWORDS = {
        "gaming":      [r"gaming", r"game", r"\brtx\b", r"\bgpu\b", r"graphics card"],
        "camera":      [r"camera", r"photo", r"photography", r"\bmp\b", r"megapixel"],
        "battery":     [r"battery", r"long.?lasting", r"backup", r"endurance"],
        "lightweight": [r"light\b", r"thin\b", r"portable", r"slim\b", r"compact"],
        "anc":         [r"anc\b", r"noise.?cancel", r"noise.?reduc"],
        "coding":      [r"coding", r"programming", r"developer", r"software"],
        "student":     [r"student", r"college", r"school", r"study", r"education"],
        "business":    [r"business", r"office", r"professional"],
        "fitness":     [r"fitness", r"running", r"workout", r"exercise", r"sport", r"gym"],
        "display":     [r"display", r"oled", r"amoled", r"resolution", r"\bscreen\b"],
        "audio":       [r"audio", r"sound", r"bass", r"music"],
        "waterproof":  [r"waterproof", r"water.?resist", r"swim"],
        "budget":      [r"budget", r"cheap", r"affordable", r"low.?cost"],
        "premium":     [r"premium", r"luxury", r"high.?end", r"flagship"],
        "performance": [r"performance", r"\bfast\b", r"speed", r"processor", r"\bcpu\b"],
    }

    _PRICE_PATTERN = re.compile(
        r'(?:under|below|upto|within|less\s*than|max(?:imum)?|budget(?:\s*of)?)'
        r'\s*(?:rs\.?|₹)?\s*(\d[\d,]*)\s*(k|lakh|lac|thousand)?',
        re.IGNORECASE,
    )
    _PRICE_PATTERN2 = re.compile(
        r'(\d[\d,]*)\s*(k|lakh|lac)?\s*(?:budget|limit|max(?:imum)?)',
        re.IGNORECASE,
    )

    _LLM_PROMPT = """\
Classify this product search query and return ONLY valid JSON (no extra text):

Query: "{query}"

Return JSON with these exact keys:
{{
  "product_type": "<one of: laptop, phone, tablet, headphone, smartwatch, or null>",
  "max_price": <number or null>,
  "features": ["<feature1>", "<feature2>"],
  "confidence": <0.0-1.0>
}}

Rules:
- max_price must be in INR (convert k→×1000, lakh→×100000)
- features: extract specific needs (gaming, camera, battery, student, etc.)
- confidence: how certain you are about product_type (0=unsure, 1=certain)
- If no product type is clear, set product_type to null"""

    def __init__(self, api_key: Optional[str] = None):
        self._api_key  = api_key
        self._model    = None
        self._llm_ok   = False

        if api_key:
            try:
                self._model  = _gemini_manager.get_model(api_key)
                self._llm_ok = self._model is not None
            except Exception as exc:
                logger.warning(f"QueryClassifier: LLM init failed — {exc}")

        logger.info(f"QueryClassifier: LLM={'enabled' if self._llm_ok else 'disabled (regex fallback)'}")

    # ── public ───────────────────────────────────────────────

    def classify(self, query: str) -> Dict[str, Any]:
        """Main classify entry-point."""
        if not query or not query.strip():
            return {"product_type": None, "max_price": None,
                    "features": [], "confidence": 0.0, "method": "empty"}

        if self._llm_ok:
            result = self._try_llm(query)
            if result:
                return result

        return self._regex_classify(query)

    def is_supported_type(self, pt: Optional[str]) -> bool:
        return pt in self.SUPPORTED_TYPES

    def get_friendly_limitation_message(self, pt: Optional[str]) -> str:
        if pt is None:
            return ("I couldn't identify the product type you're looking for. "
                    "I can help with: laptops 💻, phones 📱, tablets 📟, "
                    "headphones 🎧, and smartwatches ⌚.")
        return (f"Sorry, I don't have '{pt}' products in my database yet. "
                f"I currently support: {', '.join(self.SUPPORTED_TYPES)}.")

    # ── private ──────────────────────────────────────────────

    def _try_llm(self, query: str) -> Optional[Dict[str, Any]]:
        prompt = self._LLM_PROMPT.format(query=query)
        for attempt in range(MAX_RETRIES):
            try:
                resp = self._model.generate_content(
                    prompt,
                    generation_config={"temperature": 0.0},
                )
                raw  = resp.text.strip()
                data = _parse_json(raw)
                if data and isinstance(data, dict):
                    pt = data.get("product_type")
                    if isinstance(pt, str):
                        pt = pt.lower().strip()
                    data["product_type"] = pt if pt in self.SUPPORTED_TYPES else None
                    data.setdefault("max_price", None)
                    data.setdefault("features",  [])
                    data.setdefault("confidence", 0.8)
                    data["method"] = "llm"
                    return data
            except Exception as exc:
                err_str = str(exc)
                if "429" in err_str or "quota" in err_str.lower():
                    logger.warning("QueryClassifier: 429 quota hit — disabling LLM")
                    self._llm_ok = False
                    self._model  = None
                    return None
                logger.warning(f"QueryClassifier LLM attempt {attempt+1}: {exc}")
                time.sleep(1)
        return None

    def _extract_price(self, query: str) -> Optional[float]:
        """Extract max price from query text."""
        _MULT = {"k": 1000, "thousand": 1000, "lakh": 100000, "lac": 100000}
        for pat in (self._PRICE_PATTERN, self._PRICE_PATTERN2):
            m = pat.search(query)
            if m:
                val    = float(m.group(1).replace(",", ""))
                suffix = (m.group(2) or "").lower()
                return val * _MULT.get(suffix, 1)
        return None

    def _extract_features(self, query: str) -> List[str]:
        q = query.lower()
        return [feat for feat, pats in self._FEATURE_KEYWORDS.items()
                if any(re.search(p, q) for p in pats)]

    def _detect_type(self, query: str) -> Optional[str]:
        q = query.lower()
        for ptype, pats in self._TYPE_KEYWORDS:
            if any(re.search(p, q) for p in pats):
                return ptype
        return None

    def _regex_classify(self, query: str) -> Dict[str, Any]:
        pt       = self._detect_type(query)
        price    = self._extract_price(query)
        features = self._extract_features(query)
        conf     = 0.7 if pt else 0.2
        return {
            "product_type": pt,
            "max_price":    price,
            "features":     features,
            "confidence":   conf,
            "method":       "regex",
        }


# ── shared JSON parser (used by Cell 4 too) ──────────────────
def _parse_json(text: str) -> Optional[Dict]:
    """Parse JSON from LLM response, handling fences and truncation."""
    if not text:
        return None
    text = re.sub(r'^```json\s*', '', text.strip())
    text = re.sub(r'```\s*$',     '', text.strip()).strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        # Fix truncated JSON
        text = re.sub(r',\s*$', '', text)
        text += ']' * (text.count('[') - text.count(']'))
        text += '}' * (text.count('{') - text.count('}'))
        try:
            return json.loads(text)
        except Exception:
            return None


print("✅ Cell 3 complete  —  QueryClassifier & _parse_json defined")


✅ Cell 3 complete  —  QueryClassifier & _parse_json defined


In [5]:
# ============================================================
# CELL 4: ResponseParser & ResultFormatter
# ============================================================

class ResponseParser:
    """Parse JSON from LLM response text."""

    @staticmethod
    def parse(text: str) -> Optional[Dict]:
        """Wrapper around shared _parse_json."""
        return _parse_json(text)


class ResultFormatter:
    """Format retrieved products for display and LLM prompts."""

    EMOJI = {
        "laptop":     "💻",
        "phone":      "📱",
        "tablet":     "📟",
        "headphone":  "🎧",
        "smartwatch": "⌚",
    }

    @staticmethod
    def _price_str(price_raw) -> str:
        """Format price as ₹X,XXX."""
        try:
            if isinstance(price_raw, str):
                price_raw = price_raw.replace(",", "").replace("₹", "").strip()
            p = float(price_raw)
            return f"₹{p:,.0f}"
        except (ValueError, TypeError):
            return str(price_raw)

    @staticmethod
    def _product_name(meta: Dict[str, Any]) -> str:
        """Construct product name from brand + model (no 'name' field in DB)."""
        brand = meta.get("brand", "").strip()
        model = meta.get("model", "").strip()
        if brand and model:
            return f"{brand} {model}"
        return brand or model or "Unknown Product"

    @classmethod
    def format_for_llm(cls, results: List[Dict[str, Any]]) -> str:
        """Convert KB results to human-readable text for LLM prompt context."""
        if not results:
            return "No products found in the database."
        lines = []
        for i, r in enumerate(results, 1):
            meta  = r.get("metadata", {})
            name  = cls._product_name(meta)
            price = cls._price_str(meta.get("price", "N/A"))
            score = r.get("score", 0)
            ptype = meta.get("product_type", "")

            lines.append(f"{i}. {name} ({ptype})")
            lines.append(f"   Price: {price}  |  Relevance: {score:.2f}")

            # Extra specs
            spec_keys = ["processor", "ram", "storage", "display",
                         "camera", "battery", "category"]
            specs = {k: meta[k] for k in spec_keys if meta.get(k)}
            if specs:
                spec_str = "  |  ".join(f"{k.title()}: {v}" for k, v in specs.items())
                lines.append(f"   Specs: {spec_str}")
            lines.append("")
        return "\n".join(lines)

    @classmethod
    def display(cls, response: Dict[str, Any]) -> None:
        """Pretty CLI display of agent response."""
        if not response:
            print("⚠️  No response to display.")
            return

        # Header
        query   = response.get("query", "")
        ptype   = response.get("product_type", "")
        emoji   = cls.EMOJI.get(ptype, "🔍")
        method  = response.get("method", "")
        price   = response.get("max_price")
        results = response.get("results", [])

        print("\n" + "=" * 60)
        print(f"{emoji}  Query  : {query}")
        if ptype:
            print(f"   Type   : {ptype}")
        if price:
            print(f"   Budget : {cls._price_str(price)}")
        print(f"   Source : {method}")
        print("=" * 60)

        # LLM narrative if available
        narrative = response.get("narrative") or response.get("answer")
        if narrative:
            print(f"\n📝 {narrative}\n")

        # Product list
        if results:
            print(f"📦 Top {len(results)} Product(s) Found:\n")
            for i, r in enumerate(results, 1):
                meta  = r.get("metadata", {})
                name  = cls._product_name(meta)
                price_r = cls._price_str(meta.get("price", "N/A"))
                score = r.get("score", 0)
                pt_em = cls.EMOJI.get(meta.get("product_type", ""), "")
                print(f"  {i}. {pt_em} {name}")
                print(f"       Price : {price_r}")
                print(f"       Score : {score:.2f}")
                # optional specs
                for k in ["processor", "ram", "storage", "battery", "camera", "display"]:
                    if meta.get(k):
                        print(f"       {k.title():9}: {meta[k]}")
                print()
        elif not narrative:
            msg = response.get("message", "No results found.")
            print(f"\n{msg}\n")

        print("=" * 60 + "\n")


print("✅ Cell 4 complete  —  ResponseParser & ResultFormatter defined")


✅ Cell 4 complete  —  ResponseParser & ResultFormatter defined


In [6]:
# ============================================================
# CELL 5: MultiProductAgent
# ============================================================

class MultiProductAgent:
    """
    Orchestrates ProductRetriever + QueryClassifier + Gemini LLM
    to answer natural-language product queries.
    """

    _ANSWER_PROMPT = """\
You are a helpful product recommendation assistant.

USER QUERY: {query}

CLASSIFICATION:
- Product Type : {product_type}
- Max Budget   : {max_price}
- Key Features : {features}

PRODUCTS FROM DATABASE:
{products}

Based on the products above, write a concise recommendation (3-5 sentences).
Mention specific product names (brand + model), prices, and why they match the query.
Return ONLY valid JSON:
{{
  "answer": "<your recommendation paragraph>",
  "top_picks": ["<Product 1>", "<Product 2>"],
  "reason": "<one-line reason for top pick>"
}}"""

    def __init__(self):
        print("🚀 Initializing MultiProductAgent…")

        # Retriever
        try:
            self._retriever = ProductRetriever(DB_PATH)
            count = self._retriever.get_product_count()
            print(f"   ✅ ProductRetriever ready  ({count} products in DB)")
        except Exception as exc:
            print(f"   ❌ ProductRetriever failed: {exc}")
            raise

        # Classifier
        self._classifier = QueryClassifier(API_KEY if API_KEY else None)
        print(f"   ✅ QueryClassifier ready  "
              f"(LLM={'on' if self._classifier._llm_ok else 'off — regex'})")

        # Gemini model for answer generation
        self._model     = None
        self.llm_available = False
        if API_KEY:
            self._model = _gemini_manager.get_model(API_KEY)
            self.llm_available = self._model is not None
        print(f"   ✅ Answer LLM  : {'enabled' if self.llm_available else 'disabled'}")

        # Cache
        self._cache: Dict[str, Dict] = {}
        self._stats = {"queries": 0, "cache_hits": 0, "llm_answers": 0,
                       "db_answers": 0, "errors": 0}

        print("✅ MultiProductAgent ready\n")

    # ── helpers ──────────────────────────────────────────────

    def search_kb(
        self,
        query:        str,
        max_price:    Optional[float] = None,
        product_type: Optional[str]   = None,
    ) -> List[Dict[str, Any]]:
        """Search the knowledge base, optionally with price filter."""
        pt = product_type or self._retriever.detect_product_type(query)
        if max_price:
            return self._retriever.filter_by_price(query, max_price,
                                                   n_results=5, product_type=pt)
        return self._retriever.search(query, n_results=5, product_type=pt)

    def _try_llm(
        self,
        query:          str,
        classification: Dict[str, Any],
        kb_results:     List[Dict[str, Any]],
    ) -> Optional[Dict[str, Any]]:
        """Generate LLM-formatted answer. Returns None on failure/quota."""
        if not self.llm_available or not self._model:
            return None

        products_text = ResultFormatter.format_for_llm(kb_results)
        pt       = classification.get("product_type") or "product"
        mp       = classification.get("max_price")
        feats    = ", ".join(classification.get("features", [])) or "none specified"
        mp_str   = f"₹{mp:,.0f}" if mp else "no limit"

        prompt = self._ANSWER_PROMPT.format(
            query        = query,
            product_type = pt,
            max_price    = mp_str,
            features     = feats,
            products     = products_text,
        )

        for attempt in range(MAX_RETRIES):
            try:
                resp = self._model.generate_content(
                    prompt,
                    generation_config={"temperature": 0.2},
                )
                data = _parse_json(resp.text.strip())
                if data and isinstance(data, dict):
                    return data
            except Exception as exc:
                err_str = str(exc)
                if "429" in err_str or "quota" in err_str.lower():
                    logger.warning("MultiProductAgent: 429 quota — disabling LLM")
                    self.llm_available = False
                    self._model        = None
                    return None
                logger.warning(f"MultiProductAgent LLM attempt {attempt+1}: {exc}")
                time.sleep(1)
        return None

    def _db_response(
        self,
        query:          str,
        classification: Dict[str, Any],
        kb_results:     List[Dict[str, Any]],
    ) -> Dict[str, Any]:
        """Build response dict from DB results only (no LLM)."""
        pt    = classification.get("product_type")
        mp    = classification.get("max_price")
        feats = classification.get("features", [])

        if not kb_results:
            feat_str  = (", ".join(feats) + " ") if feats else ""
            price_str = (f" under ₹{mp:,.0f}") if mp else ""
            narrative = (f"No {feat_str}{pt or 'product'} found{price_str} in our database. "
                         f"Try broadening your search.")
        else:
            names     = [ResultFormatter._product_name(r["metadata"]) for r in kb_results[:3]]
            feat_str  = (", ".join(feats) + " ") if feats else ""
            price_str = (f" under ₹{mp:,.0f}") if mp else ""
            narrative = (f"Found {len(kb_results)} {feat_str}{pt or 'product'}(s)"
                         f"{price_str}. "
                         f"Top matches: {', '.join(names)}.")

        return {
            "query":        query,
            "product_type": pt,
            "max_price":    mp,
            "results":      kb_results,
            "narrative":    narrative,
            "method":       "database",
        }

    # ── main API ─────────────────────────────────────────────

    def ask(self, user_query: str) -> Dict[str, Any]:
        """Main entry-point: classify → search → format → return."""
        self._stats["queries"] += 1
        cache_key = user_query.lower().strip()

        # Cache check
        if cache_key in self._cache:
            self._stats["cache_hits"] += 1
            logger.info(f"Cache hit for: {user_query!r}")
            return self._cache[cache_key]

        try:
            # 1. Classify
            classification = self._classifier.classify(user_query)
            pt = classification.get("product_type")
            logger.info(f"Classified: type={pt}, "
                        f"price={classification.get('max_price')}, "
                        f"method={classification.get('method')}")

            # 2. Unsupported type guard
            if not self._classifier.is_supported_type(pt):
                msg = self._classifier.get_friendly_limitation_message(pt)
                result = {
                    "query":        user_query,
                    "product_type": pt,
                    "max_price":    None,
                    "results":      [],
                    "message":      msg,
                    "method":       "unsupported",
                }
                self._cache[cache_key] = result
                return result

            # 3. Search KB
            kb_results = self.search_kb(
                user_query,
                max_price    = classification.get("max_price"),
                product_type = pt,
            )
            logger.info(f"KB returned {len(kb_results)} result(s)")

            # 4. Try LLM answer
            llm_data = self._try_llm(user_query, classification, kb_results)
            if llm_data:
                self._stats["llm_answers"] += 1
                result = {
                    "query":        user_query,
                    "product_type": pt,
                    "max_price":    classification.get("max_price"),
                    "results":      kb_results,
                    "narrative":    llm_data.get("answer", ""),
                    "top_picks":    llm_data.get("top_picks", []),
                    "reason":       llm_data.get("reason", ""),
                    "method":       "llm",
                }
            else:
                # 5. DB-only fallback
                self._stats["db_answers"] += 1
                result = self._db_response(user_query, classification, kb_results)

            self._cache[cache_key] = result
            return result

        except Exception as exc:
            self._stats["errors"] += 1
            logger.error(f"MultiProductAgent.ask error: {exc}", exc_info=True)
            return {
                "query":   user_query,
                "message": f"An error occurred: {exc}",
                "results": [],
                "method":  "error",
            }

    def get_stats(self) -> Dict[str, Any]:
        return {
            **self._stats,
            "cache_size":    len(self._cache),
            "db_products":   self._retriever.get_product_count(),
            "llm_available": self.llm_available,
        }


print("✅ Cell 5 complete  —  MultiProductAgent defined")


✅ Cell 5 complete  —  MultiProductAgent defined


In [7]:
# ============================================================
# CELL 6: CLI & Launch
# ============================================================

EXIT_COMMANDS = {"exit", "quit", "q", "bye", "end", "stop"}
print("your exit commands are:", EXIT_COMMANDS)
HELP_TEXT = """
╔══════════════════════════════════════════════════════════╗
║          Multi-Product AI Search Agent  —  Help          ║
╠══════════════════════════════════════════════════════════╣
║  Just type a natural-language query, e.g.:               ║
║    gaming laptop under 80000                             ║
║    best camera phone below 30k                           ║
║    noise cancelling headphones under 5000                ║
║    smartwatch with fitness tracking                      ║
║                                                          ║
║  Special commands:                                       ║
║    types   — list supported product types                ║
║    stats   — show session statistics                     ║
║    help    — show this help message                      ║
║    exit / quit / q  — quit the agent                     ║
╚══════════════════════════════════════════════════════════╝
"""
print(HELP_TEXT)
def run_cli(agent: MultiProductAgent) -> None:
    """Interactive CLI loop."""
    print("\n" + "=" * 60)
    print("  🛍️   Multi-Product AI Search Agent  🛍️")
    print("=" * 60)
    print("  Supports: laptops 💻  phones 📱  tablets 📟")
    print("            headphones 🎧  smartwatches ⌚")
    print("  Type 'help' for usage  |  'exit' to quit")
    print("=" * 60 + "\n")

    while True:
        try:
            raw = input("🔍 Your query: ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\n\n👋 Interrupted — goodbye!")
            break

        if not raw:
            continue

        cmd = raw.lower()

        # ── built-in commands ────────────────────────────────
        if cmd in EXIT_COMMANDS:
            print("\n👋 Thank you for using the Multi-Product Agent. Goodbye!\n")
            break

        if cmd == "help":
            print(HELP_TEXT)
            continue

        if cmd == "types":
            types = agent._retriever.get_available_types()
            emojis = {"laptop":"💻","phone":"📱","tablet":"📟",
                      "headphone":"🎧","smartwatch":"⌚"}
            print("\n📋 Supported product types:")
            for t in types:
                print(f"   {emojis.get(t,'')} {t}")
            print()
            continue

        if cmd == "stats":
            s = agent.get_stats()
            print("\n📊 Session Stats:")
            for k, v in s.items():
                print(f"   {k:20}: {v}")
            print()
            continue

        # ── search query ─────────────────────────────────────
        print(f"\n⏳ Searching…")
        response = agent.ask(raw)
        ResultFormatter.display(response)


print("✅ Cell 6 complete  —  CLI & Launch defined")

# ── Bootstrap ────────────────────────────────────────────────
try:
    agent = MultiProductAgent()
except Exception as exc:
    print(f"\n❌ Failed to initialize agent: {exc}")
    print("   Check DB_PATH and that ChromaDB collection 'products' exists.")
    raise

# Print initial stats
s = agent.get_stats()
print(f"📊 Database: {s['db_products']} products  |  "
      f"LLM: {'enabled ✅' if s['llm_available'] else 'regex fallback ⚠️'}\n")

# Launch CLI
run_cli(agent)

# Final stats on exit
print("\n📊 Final Session Stats:")
for k, v in agent.get_stats().items():
    print(f"   {k:20}: {v}")


your exit commands are: {'q', 'bye', 'exit', 'quit', 'end', 'stop'}

╔══════════════════════════════════════════════════════════╗
║          Multi-Product AI Search Agent  —  Help          ║
╠══════════════════════════════════════════════════════════╣
║  Just type a natural-language query, e.g.:               ║
║    gaming laptop under 80000                             ║
║    best camera phone below 30k                           ║
║    noise cancelling headphones under 5000                ║
║    smartwatch with fitness tracking                      ║
║                                                          ║
║  Special commands:                                       ║
║    types   — list supported product types                ║
║    stats   — show session statistics                     ║
║    help    — show this help message                      ║
║    exit / quit / q  — quit the agent                     ║
╚══════════════════════════════════════════════════════════╝

✅ Cell 6 comple

2026-06-30 10:59:54,136 [WARNING] huggingface_hub.utils._http: Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-06-30 10:59:54,163 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/modules.json "HTTP/1.1 200 OK"
2026-06-30 10:59:54,425 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-06-30 10:59:54,449 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-06-30 10:59:54,704 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentenc

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6867.49it/s]


2026-06-30 10:59:56,494 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
2026-06-30 10:59:56,754 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-06-30 10:59:57,011 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/video_preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-06-30 10:59:57,272 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-06-30 10:59:57,543 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-06-30 10:59:57,569 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/a

🔍 Your query:  laptop under 80000



⏳ Searching…
2026-06-30 11:00:16,122 [INFO] MultiProductAgent: Classified: type=laptop, price=80000, method=llm
2026-06-30 11:00:16,148 [INFO] MultiProductAgent: KB returned 5 result(s)

💻  Query  : laptop under 80000
   Type   : laptop
   Budget : ₹80,000
   Source : llm

📝 For a laptop under ₹80,000, the Lenovo IdeaPad 5 Pro is an excellent choice at ₹80,000, featuring a powerful AMD Ryzen 7 processor, 16GB RAM, and a 512GB SSD, making it ideal for content creation and demanding tasks. If you're looking for a more budget-friendly option with good storage, the ASUS VivoBook 14 at ₹48,000 offers an Intel i5 processor and a 512GB SSD. Both options fit your specified budget and provide strong value.

📦 Top 5 Product(s) Found:

  1. 💻 Lenovo ThinkBook 14
       Price : ₹48,000
       Score : 0.53
       Processor: Intel i5-1240P
       Ram      : 8GB
       Storage  : 256GB SSD
       Battery  : 52
       Display  : 14 FHD IPS

  2. 💻 Lenovo ThinkPad E15
       Price : ₹45,000
       Sco

🔍 Your query:  quit



👋 Thank you for using the Multi-Product Agent. Goodbye!


📊 Final Session Stats:
   queries             : 1
   cache_hits          : 0
   llm_answers         : 1
   db_answers          : 0
   errors              : 0
   cache_size          : 1
   db_products         : 90
   llm_available       : True
